# GUI screenshot for the GitHub README

Generates `docs/fig_gui.png`, an annotated screenshot of the
`sat_tile_stack.labeling` browser GUI (the `lakelabel` command),
embedded in the **Labeling GUI** section of `README.md`.

How it works: load one per-lake composite NetCDF, render a single
Sentinel-2 RGB frame, embed that frame into a self-contained HTML mock
of the GUI layout, screenshot with Playwright, then overlay (a)-(g)
panel annotations with matplotlib. Output goes to `../docs/fig_gui.png`.

Adapted from `essd/notebooks/essd_fig5_gui.ipynb`; the HTML/CSS mock is
the same so the GUI screenshot stays in sync with the paper Figure 5.

In [1]:
import os
import time
import tempfile
import base64
import shutil
import io
import warnings
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import xarray as xr

warnings.filterwarnings('ignore', category=UserWarning)

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 9,
    'font.family': 'sans-serif',
    'axes.linewidth': 0.6,
})

OUT_DIR = Path('../docs').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_PATH = OUT_DIR / 'fig_gui_screenshot.png'   # intermediate
FINAL_PATH      = OUT_DIR / 'fig_gui.png'              # README references this

CANDIDATES = [
    os.environ.get('SAT_TILE_STACK_EXAMPLE_NC'),
    str(Path.home() / 'stanford_gp/research/lakes/2026/lake-vision/'
        'datasets/processed/CW2019_1524.nc'),
    '/tmp/CW2019_1889.nc',
    '/Volumes/groups/cyaolai/JoshRines/sherlock/sherlock_lakevision/'
    'composites/CW_2019/CW2019_1889.nc',
]
COMPOSITE = next((Path(p) for p in CANDIDATES if p and Path(p).exists()), None)
if COMPOSITE is None:
    raise FileNotFoundError(
        'no per-lake composite found in CANDIDATES; mount OAK or set '
        '$SAT_TILE_STACK_EXAMPLE_NC to a v1 composite (.nc with '
        "channels=['red','green','blue','mask',...] and water_area(time)).")

LAKE_ID = COMPOSITE.stem
print(f'composite: {COMPOSITE}  (LAKE_ID={LAKE_ID})')
print(f'output:    {FINAL_PATH}')


composite: /Users/jrines/stanford_gp/research/lakes/2026/lake-vision/datasets/processed/CW2019_1524.nc  (LAKE_ID=CW2019_1524)
output:    /Users/jrines/stanford_gp/research/lakes/2026/sat-tile-stack/docs/fig_gui.png


In [2]:
from playwright.async_api import async_playwright

# --- 1. Render one S2 RGB frame (mid-melt-season) from the composite ---
ds = xr.open_dataset(COMPOSITE)
times = ds.time.values
target = np.datetime64('2019-06-11')
frame_idx = int(np.argmin(np.abs(times - target)))
frame_date = np.datetime_as_string(times[frame_idx], unit='D')
n_frames = len(times)
slider_pct = 100.0 * frame_idx / max(1, n_frames - 1)
print(f'Frame {frame_idx+1}/{n_frames}: {frame_date} (slider at {slider_pct:.0f}%)')

imagery = ds.imagery.isel(time=frame_idx)
ch_names = [str(c) for c in ds.channel.values]
rgb_idx = [ch_names.index(c) for c in ('red', 'green', 'blue')]
rgb = imagery.values[rgb_idx]

scale, gamma = 10000.0, 1.4
rgb_display = np.stack([rgb[i] / scale for i in range(3)], axis=-1)
rgb_display = np.clip(rgb_display, 0, 1) ** (1.0 / gamma)
rgb_display = np.nan_to_num(rgb_display, nan=0.0)

fig_tmp, ax_tmp = plt.subplots(figsize=(5, 5))
fig_tmp.patch.set_facecolor('black')
ax_tmp.set_facecolor('black')
ax_tmp.imshow(rgb_display)
mask = imagery.sel(channel='mask').values
if not np.isnan(mask).all():
    ax_tmp.contour(mask, levels=[0.5], colors='red', linewidths=1)
ax_tmp.axis('off')
fig_tmp.subplots_adjust(left=0, right=1, top=1, bottom=0)
buf = io.BytesIO()
fig_tmp.savefig(buf, format='png', dpi=120, bbox_inches='tight',
                facecolor='black', edgecolor='none')
plt.close(fig_tmp)
buf.seek(0)
frame_b64 = base64.b64encode(buf.read()).decode()
ds.close()
print(f'Frame rendered ({len(frame_b64)//1000}KB)')

# --- 2. Self-contained HTML mock of the GUI layout (1400 x 850) ---
standalone_css = '''
* { margin: 0; padding: 0; box-sizing: border-box; }
body {
    background: #111; color: #eee; font-family: -apple-system, sans-serif;
    width: 1400px; height: 850px; position: relative;
}
#topbar {
    display: flex; align-items: center; justify-content: space-between;
    padding: 10px 20px; background: #1a1a1a; border-bottom: 1px solid #333;
    font-size: 13px; height: 44px;
}
#topbar .id { font-weight: bold; font-size: 15px; }
#topbar .progress { color: #888; white-space: nowrap; }
#progress-bar {
    width: 100px; height: 8px; background: #333; border-radius: 4px;
    overflow: hidden; display: inline-block; vertical-align: middle; margin: 0 8px;
}
#progress-fill { height: 100%; background: #4488cc; }
#right-panel {
    position: absolute; top: 44px; right: 0; bottom: 0;
    width: 340px; padding: 10px 18px;
    background: #1a1a1a; border-left: 1px solid #333;
}
#left-panel {
    position: absolute; top: 44px; left: 0; right: 340px; bottom: 0;
    padding: 8px 14px; display: flex; flex-direction: column;
}
#image-container {
    flex: 1; min-height: 0; display: flex;
    align-items: center; justify-content: center;
}
#main-image { height: 100%; aspect-ratio: 1 / 1; object-fit: cover; }
#date-label { font-size: 18px; color: #ddd; font-weight: bold; margin: 8px 0 4px; text-align: center; }
#frame-slider-container { width: 100%; padding: 6px 0 8px; }
.fake-slider { width: 100%; height: 22px; position: relative; }
.fake-slider-track {
    position: absolute; top: 6px; left: 0; right: 0;
    height: 10px; background: #444; border: 1px solid #555; border-radius: 5px;
}
.fake-slider-fill {
    position: absolute; top: 0; left: 0;
    height: 100%; background: #4488cc; border-radius: 5px;
}
.fake-slider-thumb {
    position: absolute; top: 1px;
    transform: translateX(-50%);
    width: 20px; height: 20px; border-radius: 50%;
    background: #4488cc; border: 2px solid #fff;
    box-shadow: 0 0 4px rgba(0,0,0,0.8);
}
.prob-row { display: flex; align-items: center; margin: 3px 0; font-size: 12px; }
.prob-label { width: 28px; font-weight: bold; text-align: right; margin-right: 6px; }
.prob-slider-wrap { flex: 1; position: relative; height: 24px; display: flex; align-items: center; }
.prob-slider { width: 100%; accent-color: #4488cc; margin: 0; }
.prob-ticks {
    position: absolute; bottom: 1px; left: 7px; right: 7px; height: 6px;
    display: flex; justify-content: space-between; align-items: center; pointer-events: none;
}
.prob-tick { width: 6px; height: 6px; border-radius: 50%; background: #666; border: 1px solid #888; }
.prob-value { width: 36px; text-align: center; color: #aaa; font-size: 11px; }
.section-label { font-size: 11px; color: #666; margin: 8px 0 2px; text-transform: uppercase; }
.btn-row { display: flex; gap: 6px; margin: 4px 0; }
.btn { flex: 1; padding: 8px; border: none; border-radius: 4px; font-size: 12px; font-weight: bold; color: white; }
.btn-submit { background: #228833; }
.btn-skip { background: #aa7700; }
.btn-back { background: #555; }
.btn-flag { background: #664; border: 1px solid #886; }
.sample-item { padding: 2px 6px; border-radius: 3px; white-space: nowrap; }
.sample-item.active { background: #2a4a6a; font-weight: bold; }
'''

html_top = f'''<!DOCTYPE html><html><head><meta charset="utf-8">
<style>{standalone_css}</style></head>
<body>
<div id="topbar">
  <div><span class="id">{LAKE_ID}</span></div>
  <div style="font-weight:bold;font-size:13px;padding:4px 14px;border-radius:4px;background:#1f4d2b;color:#6fdc8c;">LABEL NOW</div>
  <div class="progress"><span>687</span>/<span>954</span> labeled
    <div id="progress-bar"><div id="progress-fill" style="width:72%"></div></div>
    <span>267</span> remaining</div>
</div>
<div id="left-panel">
  <div id="image-container">
    <img id="main-image" src="data:image/png;base64,{frame_b64}" /></div>
  <div id="date-label">{frame_date} | frame {frame_idx+1}/{n_frames}</div>
  <div id="frame-slider-container">
    <div class="fake-slider">
      <div class="fake-slider-track">
        <div class="fake-slider-fill" style="width:{slider_pct:.1f}%"></div>
      </div>
      <div class="fake-slider-thumb" style="left:{slider_pct:.1f}%"></div>
    </div>
  </div>
</div>
<div id="right-panel">
  <div class="section-label" style="display:flex;justify-content:space-between;">Class probabilities
    <span style="font-size:14px;color:#4488cc;text-transform:none;">? Guide</span></div>
  <div id="prob-sliders">'''

html_controls = '''
    <div class="prob-row"><span class="prob-label">ND</span><div class="prob-slider-wrap"><input type="range" class="prob-slider" min="0" max="1" step="0.25" value="0" /><div class="prob-ticks"><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div></div></div><span class="prob-value">0</span></div>
    <div class="prob-row"><span class="prob-label">HF</span><div class="prob-slider-wrap"><input type="range" class="prob-slider" min="0" max="1" step="0.25" value="0.75" /><div class="prob-ticks"><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div></div></div><span class="prob-value">0.75</span></div>
    <div class="prob-row"><span class="prob-label">MD</span><div class="prob-slider-wrap"><input type="range" class="prob-slider" min="0" max="1" step="0.25" value="0.25" /><div class="prob-ticks"><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div></div></div><span class="prob-value">0.25</span></div>
    <div class="prob-row"><span class="prob-label">LD</span><div class="prob-slider-wrap"><input type="range" class="prob-slider" min="0" max="1" step="0.25" value="0" /><div class="prob-ticks"><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div></div></div><span class="prob-value">0</span></div>
    <div class="prob-row"><span class="prob-label">CD</span><div class="prob-slider-wrap"><input type="range" class="prob-slider" min="0" max="1" step="0.25" value="0" /><div class="prob-ticks"><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div><div class="prob-tick"></div></div></div><span class="prob-value">0</span></div>
  </div>
  <div style="text-align:center;font-size:16px;font-weight:bold;color:#33ff33;margin:6px 0;">HF</div>
  <div class="section-label">Notes</div>
  <input type="text" value="rapid drainage, fracture visible" style="width:100%;background:#222;border:1px solid #444;color:#eee;padding:4px 8px;border-radius:4px;font-size:12px;margin:4px 0;" />
  <div class="btn-row" style="margin-top:8px;"><button class="btn btn-back">&#8592; Back</button><button class="btn btn-skip">Skip &#8594;</button></div>
  <div class="btn-row"><button class="btn btn-submit">&#10003; Submit &amp; Next</button></div>
  <div class="btn-row"><button class="btn btn-flag">&#x1F6A9; Flag for revisit</button></div>
  <div class="section-label" style="margin-top:12px;">Distribution</div>
  <div style="text-align:center;margin:6px 0;"><canvas id="pie-canvas" width="160" height="160"></canvas></div>
  <div class="section-label" style="margin-top:8px;">Sample list</div>
  <select style="width:100%;background:#222;color:#eee;border:1px solid #444;border-radius:4px;padding:3px 6px;font-size:11px;margin-bottom:4px;"><option>All</option></select>
  <div style="max-height:140px;overflow-y:auto;font-size:11px;border:1px solid #333;border-radius:4px;padding:2px;">
    <div class="sample-item" style="color:#888;">&#x1F7E2; CW2019_1885</div>
    <div class="sample-item" style="color:#888;">&#x1F7E2; CW2019_1886</div>
    <div class="sample-item" style="color:#888;">&#x1F7E2; CW2019_1887</div>
    <div class="sample-item" style="color:#888;">&#x1F7E2; CW2019_1888</div>
    <div class="sample-item active">&#x1F7E1; CW2019_1889</div>
    <div class="sample-item" style="color:#888;">&#x1F7E1; CW2019_1890</div>
    <div class="sample-item" style="color:#888;">&#x1F7E1; CW2019_1891</div>
  </div>
</div>'''

js_pie = '''<script>
var c=document.getElementById("pie-canvas"),x=c.getContext("2d"),d=window.devicePixelRatio||1;
c.width=220*d;c.height=220*d;c.style.width="220px";c.style.height="220px";x.scale(d,d);
var cn=[137,112,130,200,108],nm=["ND","HF","MD","LD","CD"],
co=["#e8524a","#4caf50","#42a5f5","#ff9800","#ab47bc"],tot=687,sa=-Math.PI/2;
for(var i=0;i<5;i++){var sl=cn[i]/tot*2*Math.PI;x.beginPath();x.moveTo(110,110);
x.arc(110,110,55,sa,sa+sl);x.fillStyle=co[i];x.fill();
var m=sa+sl/2,lx=110+75*Math.cos(m),ly=110+75*Math.sin(m);
x.fillStyle="#ccc";x.font="10px sans-serif";x.textAlign="center";
x.fillText(nm[i]+"("+cn[i]+") "+Math.round(cn[i]/tot*100)+"%",lx,ly);sa+=sl;}
</script></body></html>'''

standalone = html_top + html_controls + js_pie

tmp_html = Path(tempfile.mktemp(suffix='.html'))
tmp_html.write_text(standalone)

async def take_screenshot():
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        page = await browser.new_page(viewport={'width': 1800, 'height': 1000})
        await page.goto(f'file://{tmp_html}')
        await page.wait_for_timeout(2000)
        box = await page.locator('body').bounding_box()
        await page.screenshot(
            path=str(SCREENSHOT_PATH.resolve()),
            clip={'x': box['x'], 'y': box['y'],
                  'width': box['width'], 'height': box['height']},
        )
        await browser.close()

await take_screenshot()
tmp_html.unlink()
print(f'Screenshot saved: {SCREENSHOT_PATH} ({SCREENSHOT_PATH.stat().st_size:,} bytes)')


Frame 42/153: 2019-06-11 (slider at 27%)
Frame rendered (868KB)
Screenshot saved: /Users/jrines/stanford_gp/research/lakes/2026/sat-tile-stack/docs/fig_gui_screenshot.png (768,158 bytes)


In [3]:
# Overlay (a)-(g) panel callouts and save the final PNG.
ANNOTATIONS = [
    {'label': '(a) Sentinel-2 RGB frame viewer\n     (scroll to scrub through time)',
     'xy': (0.38, 0.40), 'xytext': (0.08, 0.10), 'ha': 'left'},
    {'label': '(b) Date and frame timeline slider',
     'xy': (0.38, 0.94), 'xytext': (0.08, 0.94), 'ha': 'left'},
    {'label': '(c) Per-class probability sliders\n     (0.25 step, must sum to 1.0)',
     'xy': (0.86, 0.12), 'xytext': (0.58, 0.05), 'ha': 'left'},
    {'label': '(d) Argmax class label',
     'xy': (0.86, 0.27), 'xytext': (0.58, 0.27), 'ha': 'left'},
    {'label': '(e) Submit / skip / flag controls',
     'xy': (0.86, 0.44), 'xytext': (0.58, 0.49), 'ha': 'left'},
    {'label': '(f) Progress distribution\n     (pie chart by class)',
     'xy': (0.86, 0.63), 'xytext': (0.58, 0.70), 'ha': 'left'},
    {'label': '(g) Sample list\n     (filterable)',
     'xy': (0.86, 0.88), 'xytext': (0.58, 0.90), 'ha': 'left'},
]

img = Image.open(SCREENSHOT_PATH)
img_arr = np.asarray(img)
h, w = img_arr.shape[:2]
print(f'Screenshot: {w} x {h} px')

fig_w = 12
fig_h = fig_w * (h / w)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.imshow(img_arr)
ax.set_axis_off()

for ann in ANNOTATIONS:
    xy_px = (ann['xy'][0] * w, ann['xy'][1] * h)
    xytext_px = (ann['xytext'][0] * w, ann['xytext'][1] * h)
    ax.annotate(
        ann['label'],
        xy=xy_px, xytext=xytext_px,
        fontsize=9, fontweight='bold', color='white',
        ha=ann['ha'], va='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.75,
                  edgecolor='white', linewidth=0.5),
        arrowprops=dict(arrowstyle='-|>', color='white', lw=1.5,
                        connectionstyle='arc3,rad=0.12'),
    )

fig.tight_layout(pad=0.2)
fig.savefig(FINAL_PATH, bbox_inches='tight')
plt.close(fig)
print(f'Saved {FINAL_PATH} ({FINAL_PATH.stat().st_size:,} bytes)')

# Clean up the intermediate raw screenshot to keep docs/ tidy
try:
    SCREENSHOT_PATH.unlink()
except FileNotFoundError:
    pass


Screenshot: 1400 x 850 px
Saved /Users/jrines/stanford_gp/research/lakes/2026/sat-tile-stack/docs/fig_gui.png (3,449,389 bytes)
